<a href="https://colab.research.google.com/github/Irfan-code-cloud/ML-Internship-flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
BASE_DATA_PATH = "hf://datasets/FlyRank/internship-warehouse"

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
feature_vector_query = f"""
SELECT
    report_date AS date,
    client_hash_id,
    content_hash_id,

    -- Feature 1 & 2: Lagged Search Metrics (7-day rolling averages)
    AVG(gsc_avg_position) OVER (
        PARTITION BY content_hash_id ORDER BY report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS gsc_position_lag7,

    AVG(gsc_impressions) OVER (
        PARTITION BY content_hash_id ORDER BY report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS gsc_impressions_lag7,

    -- Feature 3: Lagged On-Page Engagement
    AVG(ga4_total_engagement_sec) OVER (
        PARTITION BY content_hash_id ORDER BY report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS ga4_engagement_sec_lag7,

    -- Feature 4 & 5: AI Optimization Scores & Calendar Signals
    COALESCE(ai_gemini, 0) AS ai_gemini_score,
    CASE WHEN DAYOFWEEK(report_date) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend,

    -- Target / Label
    gsc_clicks AS target_clicks

FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
"""

df_features = con.sql(feature_vector_query).to_df().dropna().reset_index(drop=True)
print(f"✅ Feature Vector Shape: {df_features.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Feature Vector Shape: (300491, 9)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Data Dictionary & Metadata

| Feature Name | Feature Meaning | Missing Value Strategy | Available Before $T_0$? |
| :--- | :--- | :--- | :--- |
| **`date`** | The specific calendar date of the performance record (`report_date`). | None; primary partition key. | Yes ($T_0$ partition date). |
| **`client_hash_id`** | Anonymized unique identifier for the client account. | None; primary key filter. | Yes (Static metadata). |
| **`content_hash_id`** | Anonymized unique identifier for the specific content piece/URL. | None; primary key filter. | Yes (Static metadata). |
| **`gsc_position_lag7`** | 7-day rolling average search ranking position ($t-7$ to $t-1$). | Dropped initial window nulls via `.dropna()`. | **Yes** (Strictly historical window $t-1$). |
| **`gsc_impressions_lag7`** | 7-day rolling average search impressions ($t-7$ to $t-1$). | Dropped initial window nulls via `.dropna()`. | **Yes** (Strictly historical window $t-1$). |
| **`ga4_engagement_sec_lag7`**| 7-day rolling average on-page engagement time in seconds ($t-7$ to $t-1$). | Dropped initial window nulls via `.dropna()`. | **Yes** (Strictly historical window $t-1$). |
| **`ai_gemini_score`** | Numerical score/flag for AI content optimization via Gemini. | Imputed with `0` using `COALESCE(ai_gemini, 0)`. | **Yes** (Available at/before publication). |
| **`is_weekend`** | Binary indicator (`1` for Saturday/Sunday, `0` for Weekday). | Deterministic; derived from calendar date. | **Yes** (Known prior to prediction). |
| **`target_clicks`** | Total daily search clicks recorded on $T_0$ (`gsc_clicks`). | Ground truth label; rows with missing target filtered. | **No** (Target Label recorded at $T_0$). |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage Hunt Validation Results

We conducted three automated tests to ensure zero data leakage across feature vectors:

1. **Target Correlation Test:** Calculated linear correlations between all candidate features and `target_clicks`. The maximum absolute correlation observed was well below the $0.90$ threshold, confirming no feature is acting as a proxy for same-day ground truth.
2. **Temporal Boundary Test:** Verified that all partition records end before `2026-06-01`. This guarantees that the sealed out-of-time test set (`month=2026-06`) remains strictly isolated.
3. **Forbidden Signal Audit:** Confirmed that raw unlagged metrics (`gsc_clicks`, `ga4_clicks`, `sessions_paid`) were completely excluded from the feature space, preventing same-day target contamination.

In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Test 1: Check Target-to-Feature Correlation
# ---------------------------------------------------------
# Compute correlation matrix between features and the target label
feature_cols = [
    'gsc_position_lag7',
    'gsc_impressions_lag7',
    'ga4_engagement_sec_lag7',
    'ai_gemini_score',
    'is_weekend'
]
corr_matrix = df_features[feature_cols + ['target_clicks']].corr()

print("🔍 Target Correlation Matrix:")
print(corr_matrix['target_clicks'].round(4))

# Assert no feature has an suspiciously high correlation (> 0.90) with the target
max_corr = corr_matrix['target_clicks'].drop('target_clicks').abs().max()
assert max_corr < 0.90, f"❌ Data Leakage Suspicion! Feature correlation with target is {max_corr}"
print(f"✅ Pass: Max feature correlation with target is {max_corr:.4f} (Threshold: < 0.90)")

# ---------------------------------------------------------
# Test 2: Temporal Boundary Verification (No Future Dates)
# ---------------------------------------------------------
# Ensure all data comes strictly from the historical training partition (March 2026)
max_date = pd.to_datetime(df_features['date']).max()
print(f"📅 Maximum Record Date in Dataset: {max_date.strftime('%Y-%m-%d')}")

# Guardrail: Ensure no June 2026 (sealed test month) or future dates leaked in
assert max_date < pd.to_datetime('2026-06-01'), "❌ Leakage Error: Future/Sealed test data present!"
print("✅ Pass: Zero lookahead temporal leakage into future/sealed months.")

# ---------------------------------------------------------
# Test 3: Same-Day Target Signal Exclusion Check
# ---------------------------------------------------------
# Verify same-day gsc_clicks or ga4_clicks are not present in feature columns
forbidden_exact_signals = ['gsc_clicks', 'ga4_clicks', 'total_clicks', 'sessions_paid']
leaked_columns = [col for col in feature_cols if col in forbidden_exact_signals]

assert len(leaked_columns) == 0, f"❌ Leakage Error: Found forbidden columns {leaked_columns}"
print("✅ Pass: No same-day target or ground-truth columns present in feature matrix.")

🔍 Target Correlation Matrix:
gsc_position_lag7         -0.1320
gsc_impressions_lag7       0.5549
ga4_engagement_sec_lag7    0.3324
ai_gemini_score            0.0135
is_weekend                -0.0226
target_clicks              1.0000
Name: target_clicks, dtype: float64
✅ Pass: Max feature correlation with target is 0.5549 (Threshold: < 0.90)
📅 Maximum Record Date in Dataset: 2026-03-31
✅ Pass: Zero lookahead temporal leakage into future/sealed months.
✅ Pass: No same-day target or ground-truth columns present in feature matrix.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

The following table details all data fields, windows, and flags that were deliberately excluded from the final feature vector matrix, along with the precise technical rationale for their omission:

| Excluded Field / Data Slice | Reason for Refusal |
| :--- | :--- |
| **`gsc_clicks` (Same-Day $T_0$)** | **Target Label Leakage:** Represents the exact daily metric being predicted; including same-day clicks directly leaks ground-truth labels. |
| **`gsc_avg_position` & `gsc_impressions` (Unlagged $T_0$)** | **Same-Day Target Proxy:** Same-day search metrics co-occur with clicks and leak active search demand recorded during the prediction window. |
| **`ga4_total_engagement_sec` (Unlagged $T_0$)** | **Post-Click Telemetry:** On-page duration occurs after a user clicks; using same-day engagement introduces severe downstream target contamination. |
| **`June 2026 Data (month=2026-06)`** | **Sealed Out-of-Time Window:** Future evaluation partition reserved exclusively for out-of-time model validation; strictly isolated from training. |
| **`client_has_gsc` / `client_has_ga4`** | **Redundant Toggles:** Row-level boolean toggles duplicated by explicit `gsc_data_available` and `ga4_data_available` filtering. |
| **`client_hash_id` & `content_hash_id`** | **High-Cardinality Identifiers:** High-cardinality string keys that risk model memorization and overfitting without providing generalizable predictive signal. |
| **`sessions_paid` / `sessions_organic`** | **Confounded Traffic Signals:** Unlagged traffic acquisition metrics that contain target-correlated noise and post-click user actions. |

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`